In [1]:
from utils import *
from plotly.subplots import make_subplots

results_dir = Path("../results/baseline")
scalars = TBScalars(".cache/baseline")

In [2]:
res_df = []
for test in results_dir.iterdir():
    env, ratio, seed = test.name.split("-")
    ratio = int(ratio.removeprefix("ratio="))
    seed = int(seed.removeprefix("seed="))
    res_df.append({"path": test, "env": env, "ratio": ratio, "seed": seed})
    scalars.read(test)
res_df = pd.DataFrame.from_records(res_df)
res_df

,path,env,ratio,seed
0,../results/baseline/CrazyClimber-ratio=8-seed=3,CrazyClimber,8,3
1,../results/baseline/Assault-ratio=4-seed=2,Assault,4,2
2,../results/baseline/Assault-ratio=32-seed=1,Assault,32,1
3,../results/baseline/CrazyClimber-ratio=32-seed=3,CrazyClimber,32,3
4,../results/baseline/Assault-ratio=8-seed=0,Assault,8,0
...,...,...,...,...
85,../results/baseline/Assault-ratio=2-seed=3,Assault,2,3
86,../results/baseline/CrazyClimber-ratio=2-seed=2,CrazyClimber,2,2
87,../results/baseline/CrazyClimber-ratio=4-seed=3,CrazyClimber,4,3
88,../results/baseline/CrazyClimber-ratio=32-seed=0,CrazyClimber,32,0


In [7]:
envs = res_df["env"].unique()

fig = make_subplots(
    rows=1,
    cols=len(envs),
    column_titles=[*envs],
)

for col, env in enumerate(envs, 1):
    dfs = []
    for _, test in res_df[res_df["env"] == env].iterrows():
        df = scalars.read(test["path"])
        df = df[df["tag"] == "val/mean_ep_ret"]
        df["index"] = np.arange(len(df))
        df["ratio"] = test["ratio"]
        dfs.append(df)
    df = pd.concat(dfs)

    g = df.groupby(["index", "ratio"])
    avg_df = pd.DataFrame.from_records(
        {
            "score_mean": g["value"].mean(),
            "score_std": g["value"].std(),
            "step": g["step"].median(),
        }
    )
    avg_df = avg_df.reset_index()

    colors = make_color_iter(palette="Dark24")
    for ratio, color in zip(avg_df["ratio"].unique(), colors):
        df = avg_df[avg_df["ratio"] == ratio]

        kw = dict(name=f"Ratio = {ratio}")
        if col != 1:
            kw.update(showlegend=False)

        for trace in err_line(
            x=df["step"],
            y=df["score_mean"],
            std=df["score_std"],
            color=color,
            **kw,
        ):
            fig.add_trace(trace, row=1, col=col)

fig